# Task 1: Sales Performance Dashboard
### Kinetrexa Software Pvt. Ltd. — Data Analytics Internship

This notebook covers: data cleaning, KPI analysis, and visualizations for the sales dataset.


## Step 1: Install & Import Libraries
Run this first — Colab already has these, this just makes sure.

In [ ]:
!pip install pandas matplotlib seaborn openpyxl -q

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)


## Step 2: Upload Your Dataset

1. Download the **"Online Retail II"** dataset from Kaggle (search "Online Retail II UCI" on kaggle.com)
2. Run the cell below — a file upload button will appear
3. Click it and select your downloaded CSV/XLSX file


In [ ]:
from google.colab import files
uploaded = files.upload()

# This will show the filename you uploaded - copy it into the next cell
print(list(uploaded.keys()))


## Step 3: Load the Data

**Important:** Replace `'your_file_name.csv'` below with the exact filename you just uploaded (shown above).
If your file is `.xlsx` instead of `.csv`, use `pd.read_excel()` instead of `pd.read_csv()`.


In [ ]:
# Replace with your actual uploaded filename
FILENAME = 'your_file_name.csv'

# If CSV:
df = pd.read_csv(FILENAME, encoding='ISO-8859-1')

# If Excel, comment the line above and uncomment this:
# df = pd.read_excel(FILENAME)

print("Shape:", df.shape)
df.head()


In [ ]:
# Check column names and data types
df.info()


## Step 4: Data Cleaning

Standard cleaning for the Online Retail II dataset. If your column names differ,
adjust the column names in the code below to match what you saw in Step 3.


In [ ]:
# Standardize column names (adjust if your dataset uses different names)
df.columns = [c.strip() for c in df.columns]
print(df.columns.tolist())


In [ ]:
# Common column names in Online Retail II — rename if needed to match your file
# Typical columns: Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country
# If your columns are named differently (e.g. 'InvoiceNo', 'UnitPrice', 'CustomerID'), rename them here:

rename_map = {
    'InvoiceNo': 'Invoice',
    'UnitPrice': 'Price',
    'CustomerID': 'Customer ID'
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

print(df.columns.tolist())


In [ ]:
# Remove cancelled orders (Invoice starting with 'C') and invalid rows
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Remove rows with non-positive Quantity or Price (returns, errors, free items)
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Drop exact duplicate rows
df = df.drop_duplicates()

# Create Revenue column
df['Revenue'] = df['Quantity'] * df['Price']

print("Cleaned shape:", df.shape)
df.head()


In [ ]:
# Save cleaned data (downloads to your computer)
df.to_csv('sales_cleaned.csv', index=False)

from google.colab import files
files.download('sales_cleaned.csv')


## Step 5: Core KPIs

In [ ]:
total_revenue = df['Revenue'].sum()
total_orders = df['Invoice'].nunique()
total_customers = df['Customer ID'].nunique() if 'Customer ID' in df.columns else None
avg_order_value = total_revenue / total_orders

print(f"Total Revenue: £{total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
if total_customers:
    print(f"Total Unique Customers: {total_customers:,}")
print(f"Average Order Value: £{avg_order_value:,.2f}")


## Step 6: Revenue Trend Over Time

In [ ]:
monthly_revenue = df.set_index('InvoiceDate').resample('M')['Revenue'].sum()

plt.figure()
monthly_revenue.plot(kind='line', marker='o', color='#2c6e91')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue (£)')
plt.tight_layout()
plt.savefig('revenue_trend.png', dpi=150)
plt.show()


## Step 7: Top 10 Products by Revenue

In [ ]:
top_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)

plt.figure()
top_products.sort_values().plot(kind='barh', color='#3d8f6f')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Revenue (£)')
plt.tight_layout()
plt.savefig('top_products.png', dpi=150)
plt.show()

top_products


## Step 8: Regional (Country-wise) Sales

In [ ]:
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

plt.figure()
country_revenue.sort_values().plot(kind='barh', color='#c26b3e')
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Revenue (£)')
plt.tight_layout()
plt.savefig('country_revenue.png', dpi=150)
plt.show()

country_revenue


## Step 9: Customer Purchasing Behavior

In [ ]:
if 'Customer ID' in df.columns:
    orders_per_customer = df.groupby('Customer ID')['Invoice'].nunique()

    plt.figure()
    orders_per_customer.plot(kind='hist', bins=30, color='#7a5ea8')
    plt.title('Distribution of Orders per Customer')
    plt.xlabel('Number of Orders')
    plt.ylabel('Number of Customers')
    plt.tight_layout()
    plt.savefig('orders_per_customer.png', dpi=150)
    plt.show()

    repeat_customers = (orders_per_customer > 1).sum()
    one_time_customers = (orders_per_customer == 1).sum()
    print(f"Repeat customers: {repeat_customers}")
    print(f"One-time customers: {one_time_customers}")


## Step 10: Download All Charts

Run this to download the chart images — you'll need them for your README and PDF report.


In [ ]:
from google.colab import files
for fname in ['revenue_trend.png', 'top_products.png', 'country_revenue.png', 'orders_per_customer.png']:
    try:
        files.download(fname)
    except Exception as e:
        print(f"Could not download {fname}: {e}")


## Step 11: Write Down Your Key Insights

Based on the numbers and charts above, fill in your own conclusions here as markdown text
(double-click this cell to edit). Example:

- Revenue peaked in [month] due to [reason]
- Top 5 products contributed X% of total revenue
- [Country] accounted for the largest share of sales
- X% of customers are repeat buyers, indicating [insight]

**Next steps:** Use `sales_cleaned.csv` (downloaded in Step 4) to build your Tableau Public dashboard,
and use these charts + insights to write your Business Insights Report (PDF).
